## Ruiz2023

In [ ]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(ggplot2)
library(dplyr)
library(glue)
library(ggtree)
library(stringr)
library(tidygraph)
library(patchwork)
library(gridExtra)
library(grid)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


ggtree v3.6.2 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan,

R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.6 (Ootpa)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] future_1.32.0      gridExtra_2.3      patchwork_1.1.2    tidygraph_1.2.3   
 [5] stringr_1.5.0      ggtree_3.6.2       glue_1.6.2         ggplot2_3.4.2     
 [9] data.table_1.14.8  Seurat_5.0.1       SeuratObject_5.0.1 sp_1.6-1          
[13] dplyr_1.1.2     

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
dmg <- subset(dmg, Study == 'Ruiz2023')
dmg
gc()

An object of class Seurat 
38576 features across 205805 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5218684,278.8,9908102,529.2,9908102,529.2
Vcells,3339325343,25477.1,8947928777,68267.3,9947389647,75892.6


In [ ]:
samples <- names(table(dmg$SampleID))
samples 

[1] "BT042_PD"                   "BT042_pons_1"              
 [3] "BT042_pons_2"               "BT072_region_1"            
 [5] "BT072_region_2"             "BT789AAQ"                  
 [7] "GNG_region_10"              "GNG_region_11"             
 [9] "GNG_region_12"              "GNG_region_6"              
[11] "T19-90627_466AAL"           "T19-90673_577AAL_section_1"
[13] "T19-90673_577AAL_section_2" "T19-91014_635AAM_core"     
[15] "T19-91014_635AAM_edge"      "T20-01237_291AAN"          
[17] "T20-90005_998AAH"           "T20-90151_212AAK"          
[19] "T20-90239_175AAA"           "T20-90296_472AAL_autopsy"  
[21] "T20-90296_472AAL_diagnosis" "T20-90296_472AAL_relapse"  
[23] "T20-90372_190AAO"           "T20-93210_698AAO_region_1" 
[25] "T20-93210_698AAO_region_2"  "T20-93369_977AAO"          
[27] "T20-93623_348AAP"           "T21-90437_071AAQ"          
[29] "T21-90517_099AAQ"           "T21-90581_212AAQ_region_1" 
[31] "T21-90581_212AAQ_region_2"  "T21-90610_232AAQ"          
[33] "T21-90717_MIMIC002"         "T21-90789_MIMIC005"        
[35] "T21-90813_MIMIC006"         "T21-90868_448AAQ"          
[37] "T21-91022_MIMIC008"         "T21-91074_MIMIC010"        
[39] "T21-91160_MIMIC013"         "T22-90003_MIMIC014"        
[41] "T22-90066_MIMIC015"         "T22-90221_148AAR"          
[43] "T22-90282_MIMIC023"         "VU248"                     
[45] "VU266"                      "VU314"                     
[47] "VU494"                      "VUMC11_pons"               
[49] "VUMC11_thalamus"            "VUMC17"                    
[51] "VUMC17_healthy"

In [ ]:
options(warn = 0)

In [ ]:
for(i in 3:length(samples)){

    nb = Numbat$new(out_dir = samples[i])
    
    p <- nb$plot_consensus()
  
    pdf(file = paste0('figures/', samples[i], "_icnv_concensus.pdf"), height = 0.5, width = 10)
    print(p)
    dev.off()
}

Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 7 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 6 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 6 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data po

## Jesssa multiome

In [ ]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(ggplot2)
library(dplyr)
library(glue)
library(ggtree)
library(stringr)
library(tidygraph)
library(patchwork)
library(gridExtra)
library(grid)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


ggtree v3.6.2 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan,

R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.6 (Ootpa)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] future_1.32.0      gridExtra_2.3      patchwork_1.1.2    tidygraph_1.2.3   
 [5] stringr_1.5.0      ggtree_3.6.2       glue_1.6.2         ggplot2_3.4.2     
 [9] data.table_1.14.8  Seurat_5.0.1       SeuratObject_5.0.1 sp_1.6-1          
[13] dplyr_1.1.2     

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
dmg <- subset(dmg, Batch_for_correction == '10Xv1_nuclei_multiome')
dmg

An object of class Seurat 
38576 features across 80155 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
samples <- names(table(dmg$SampleID))
samples <- gsub('_Multiome', '', samples)
samples 

[1] "P-1694_S-1694"  "P-1701_S-1701"  "P-1709_S-1709"  "P-1764_S-1766" 
 [5] "P-1779_S-1781"  "P-1780_S-1780"  "P-6253_S-8498"  "P-6337_S-8821" 
 [9] "P-6640_S-9581"  "P-6774_S-10146"

In [ ]:
options(warn = 0)

In [ ]:
for(i in seq_along(samples)){

    nb = Numbat$new(out_dir = paste0('multiome/', samples[i]))
    
    p <- nb$plot_consensus()
  
    pdf(file = paste0('figures/multiome_', samples[i], "_icnv_concensus.pdf"), height = 0.5, width = 10)
    print(p)
    dev.off()
}

Warning message:
“ggrepel: 4 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 4 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 1 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 7 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 8 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 1 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data po

## Jessa sc

In [ ]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(ggplot2)
library(dplyr)
library(glue)
library(ggtree)
library(stringr)
library(tidygraph)
library(patchwork)
library(gridExtra)
library(grid)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


ggtree v3.6.2 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan,

R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.6 (Ootpa)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] future_1.32.0      gridExtra_2.3      patchwork_1.1.2    tidygraph_1.2.3   
 [5] stringr_1.5.0      ggtree_3.6.2       glue_1.6.2         ggplot2_3.4.2     
 [9] data.table_1.14.8  Seurat_5.0.1       SeuratObject_5.0.1 sp_1.6-1          
[13] dplyr_1.1.2     

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
dmg <- subset(dmg, Batch_for_correction == '10Xv3_cell_rna')
dmg

An object of class Seurat 
38576 features across 57305 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
samples <- names(table(dmg$SampleID))
samples <- gsub('_RNA_only', '', samples)
samples 

[1] "P-6117_S-8370" "P-6240_S-8628" "P-6328_S-8672" "P-6337_S-8821"
[5] "P-6519_S-9084"

In [ ]:
options(warn = 0)

In [ ]:
for(i in 1:length(samples)){

     nb = Numbat$new(out_dir = paste0('sc/', samples[i]))
    
    p <- nb$plot_consensus()
  
    pdf(file = paste0('figures/scRNA_', samples[i], "_icnv_concensus.pdf"), height = 0.5, width = 10)
    print(p)
    dev.off()
}

Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 1 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 4 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data po

## Jessa sn

In [ ]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(ggplot2)
library(dplyr)
library(glue)
library(ggtree)
library(stringr)
library(tidygraph)
library(patchwork)
library(gridExtra)
library(grid)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


ggtree v3.6.2 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan,

R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.6 (Ootpa)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] future_1.32.0      gridExtra_2.3      patchwork_1.1.2    tidygraph_1.2.3   
 [5] stringr_1.5.0      ggtree_3.6.2       glue_1.6.2         ggplot2_3.4.2     
 [9] data.table_1.14.8  Seurat_5.0.1       SeuratObject_5.0.1 sp_1.6-1          
[13] dplyr_1.1.2     

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
dmg <- subset(dmg, Batch_for_correction == '10Xv3_nuclei_rna')
dmg

An object of class Seurat 
38576 features across 54529 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
samples <- names(table(dmg$SampleID))
samples <- gsub('_RNA_only', '', samples)
samples 

[1] "P-1569_S-1569" "P-1713_S-1713" "P-1741_S-2756" "P-1764_S-1766"
 [5] "P-1775_S-1775" "P-2687_S-2688" "P-3387_S-3411" "P-3407_S-3447"
 [9] "P-4198_S-4459" "P-4504_S-4916" "P-5099_S-6218" "P-5613_S-7162"
[13] "P-6251_S-8496" "P-6253_S-8498" "P-6254_S-8499" "P-6255_S-8500"
[17] "P-6640_S-9581"

In [ ]:
options(warn = 0)

In [ ]:
for(i in 1:length(samples)){

    nb = Numbat$new(out_dir = paste0('sn/', samples[i]))
    
    p <- nb$plot_consensus()
  
    pdf(file = paste0('figures/snRNA_', samples[i], "_icnv_concensus.pdf"), height = 0.5, width = 10)
    print(p)
    dev.off()
}

Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 4 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 6 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 3 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 5 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 7 unlabeled data po